# Task 1 — Load, Inspect, and Clean

In [96]:
import pandas as pd
import numpy as np

# for me, kagglehub is not installed, so I downloaded the dataset and read it using pandas
df = pd.read_csv("cardekho_dataset.csv")

df.shape

(15411, 14)

In [97]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 1.6+ MB


In [98]:
df.isnull().sum() * 100 / len(df)

Unnamed: 0           0.0
car_name             0.0
brand                0.0
model                0.0
vehicle_age          0.0
km_driven            0.0
seller_type          0.0
fuel_type            0.0
transmission_type    0.0
mileage              0.0
engine               0.0
max_power            0.0
seats                0.0
selling_price        0.0
dtype: float64

In [99]:
# Drop rows where selling_price is missing.
df.dropna(subset = ['selling_price'], inplace = True)

In [100]:
df['mileage'] = df['mileage'].astype(str).str.replace('kmpl','').str.strip()
df['mileage'] = pd.to_numeric(df['mileage'], errors = 'coerce')

In [101]:
df['engine'] = df['engine'].astype(str).str.strip()
df['engine'] = pd.to_numeric(df['engine'], errors = 'coerce')

In [102]:
df['max_power'] = df['max_power'].astype(str).str.replace('bhp','').str.strip()
df['max_power'] = pd.to_numeric(df['max_power'], errors = 'coerce')

In [103]:
# there are no null values in any feature to impute,
# if there were, the code would be like this:
# df['mileage'].fillna(df['mileage'].median(), inplace = True)
# df['engine'].fillna(df['engine'].median(), inplace = True)
# df['max_power'].fillna(df['max_power'].median(), inplace = True)

In [104]:
# Remove rows where selling_price is 999999999 or below 10000.
unnecessary = df[(df['selling_price'] == 999999999) | (df['selling_price'] < 10000)]
# there is no such columns to remove, but if there were, the code would be like this:
# df.drop(unnecessary.index, inplace = True)

In [105]:
# Drop all duplicate rows.
df.duplicated().sum() # no duplicates, if there were, the code: df.drop_duplicates(inplace = True)

0

In [106]:
df.shape

(15411, 14)

# Task 2 — Encode Categorical Features

In [107]:
# Apply label encoding to transmission_type — map Manual → 0 and Automatic → 1.
df['transmission_type'] = df['transmission_type'].map({'Manual':0, 'Automatic': 1})


In [108]:
# Apply one-hot encoding to fuel_type and seller_type using pd.get_dummies() with drop_first=True.
df = pd.get_dummies(df, columns= ['seller_type', 'fuel_type'], drop_first = True, dtype = int)

In [109]:
# Print the column list of the final DataFrame using df.columns.tolist().
column_list = df.columns.tolist()
column_list

['Unnamed: 0',
 'car_name',
 'brand',
 'model',
 'vehicle_age',
 'km_driven',
 'transmission_type',
 'mileage',
 'engine',
 'max_power',
 'seats',
 'selling_price',
 'seller_type_Individual',
 'seller_type_Trustmark Dealer',
 'fuel_type_Diesel',
 'fuel_type_Electric',
 'fuel_type_LPG',
 'fuel_type_Petrol']

In a markdown cell, answer: Why is drop_first=True used in one-hot encoding, and what does a row of all zeros in the new dummy columns represent?  
  
Using drop_first = True in one-hot encoding, helps to avoid the dummy variable trap, which is a situation where one feature can be perfectly predicted from others. By dropping one category, all other categories can determine it.  
A row of all zeros in the divided columns means that the observation belongs to the dropped (reference) category.

# Task 3 — Split and Compute Baseline MAE


In [110]:
# Define X (all columns except selling_price) and y (selling_price). Drop any remaining non-numeric columns from X.
df = df.drop(columns=['Unnamed: 0'])

In [111]:
X = df.drop(columns = ['selling_price'])
y = df['selling_price']
X = X.select_dtypes(include=['number'])
X.columns

Index(['vehicle_age', 'km_driven', 'transmission_type', 'mileage', 'engine',
       'max_power', 'seats', 'seller_type_Individual',
       'seller_type_Trustmark Dealer', 'fuel_type_Diesel',
       'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol'],
      dtype='object')

In [112]:
# Split into 80% train and 20% test using train_test_split with random_state=42.
X_np = X.values
y_np = y.values
np.random.seed(42)
indices = np.arange(len(y_np))
np.random.shuffle(indices)
split = int(0.8 * len(y_np))
train_idx = indices[:split]
test_idx = indices[split:]

In [113]:
# Compute the baseline MAE: predict the mean of y_train for every row in the test set and calculate the MAE against y_test.
X_train, X_test = X_np[train_idx], X_np[test_idx]
y_train, y_test = y_np[train_idx], y_np[test_idx]
mean_price = np.mean(y_train)
y_pred = np.full(len(y_test), mean_price)

In [114]:
#Print: "Baseline MAE: ₹X" rounded to the nearest rupee.
mae = np.mean(np.abs(y_test - y_pred))
print(f"Baseline MAE: ₹{round(mae)}")

Baseline MAE: ₹451308
